# APP Fraud Mule-Chain Demo (FR-030)

This notebook provides an additive, deterministic demo of APP fraud mule-chain detection.
It uses a mocked rapid-transfer chain and renders standardized investigator outputs.

In [ ]:
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path

import pandas as pd
import nest_asyncio

from notebook_config import (
    init_notebook,
    mask_pii,
    render_decision_block,
    render_reason_codes_block,
)
from banking.analytics.detect_mule_chains import MuleChainDetector

In [ ]:
config = init_notebook(check_env=True, check_services=False)
config

In [ ]:
detector = MuleChainDetector()
records = []

try:
    detector.connect()
    records = detector.detect_mule_chains_as_records(min_hops=3, max_hop_minutes=60)
except Exception as exc:
    print(f"Live detection unavailable: {exc}")
finally:
    detector.close()

if not records:
    print("No live alerts found. Using deterministic mock fallback (seed-42 baseline).")
    base_time = datetime(2026, 1, 15, 12, 0, tzinfo=timezone.utc)
    mock_path = {
        "accounts": ["ACC-VICTIM-8F3A", "ACC-MULE1-2B1C", "ACC-MULE2-4D5E", "ACC-EXIT-9123"],
        "transactions": [
            {
                "transaction_id": "TX-APP-001",
                "amount": 5000.00,
                "transaction_date": base_time.isoformat(),
            },
            {
                "transaction_id": "TX-APP-002",
                "amount": 4800.00,
                "transaction_date": (base_time + timedelta(minutes=8)).isoformat(),
            },
            {
                "transaction_id": "TX-APP-003",
                "amount": 4620.00,
                "transaction_date": (base_time + timedelta(minutes=19)).isoformat(),
            },
        ],
    }

    mock_alert = detector._build_alert_from_path(mock_path, min_hops=3, max_hop_minutes=60)
    if mock_alert:
        records = [
            {
                "alert_id": mock_alert.alert_id,
                "victim_account_id": mock_alert.victim_account_id,
                "mule_account_ids": mock_alert.mule_account_ids,
                "cash_out_account_id": mock_alert.cash_out_account_id,
                "hop_count": mock_alert.hop_count,
                "total_value": mock_alert.total_value,
                "average_hop_minutes": round(mock_alert.average_hop_minutes, 2),
                "risk_score": round(mock_alert.risk_score, 4),
                "reason_codes": [
                    "APP_MULE_CHAIN",
                    "RAPID_TRANSFER_VELOCITY",
                    "MULTI_HOP_CASH_OUT_PATH",
                ],
                "rationale": (
                    f"Funds traversed {mock_alert.hop_count} hops with average latency "
                    f"{mock_alert.average_hop_minutes:.1f} minutes."
                ),
                "evidence_summary": (
                    f"Victim={mock_alert.victim_account_id}; "
                    f"Mules={len(mock_alert.mule_account_ids)}; "
                    f"CashOut={mock_alert.cash_out_account_id}; "
                    f"Total=${mock_alert.total_value:,.2f}; "
                    f"Risk={mock_alert.risk_score:.3f}"
                ),
            }
        ]

records = sorted(records, key=lambda item: (item["alert_id"], -item["risk_score"]))
active_record = records[0] if records else None
active_record

In [ ]:
if active_record:
    evidence_df = pd.DataFrame(
        {
            "metric": [
                "Alert ID",
                "Victim Account",
                "Mule Path",
                "Cash-out Account",
                "Hop Count",
                "Total Value",
                "Average Hop Minutes",
                "Risk Score",
            ],
            "value": [
                active_record["alert_id"],
                mask_pii(active_record["victim_account_id"]),
                " -> ".join(mask_pii(m) for m in active_record["mule_account_ids"]),
                mask_pii(active_record["cash_out_account_id"]),
                active_record["hop_count"],
                f"${active_record['total_value']:,.2f}",
                active_record["average_hop_minutes"],
                active_record["risk_score"],
            ],
        }
    )
    display(evidence_df)
else:
    print("No mule-chain alert generated in demo scenario.")

In [ ]:
if active_record:
    decision = "BLOCK & ESCALATE" if active_record["risk_score"] >= 0.7 else "REVIEW"
    confidence = int(round(active_record["risk_score"] * 100))

    render_decision_block(
        decision=decision,
        confidence=confidence,
        action="Freeze chain accounts and escalate to Fraud Operations within SLA.",
        why=active_record["rationale"],
        evidence=active_record["evidence_summary"],
    )

    render_reason_codes_block(
        reason_codes=active_record["reason_codes"],
        rationale=(
            "Funds movement patterns match known automated APP fraud disbursement signatures."
        ),
    )

    export_dir = Path("exports/evidence/mule_chains")
    export_dir.mkdir(parents=True, exist_ok=True)
    timestamp_utc = datetime(2026, 1, 15, 12, 0, tzinfo=timezone.utc).isoformat()

    evidence_payload = {
        "alert_id": active_record["alert_id"],
        "timestamp": timestamp_utc,
        "detector": "MuleChainDetector",
        "decision": decision,
        "confidence": confidence,
        "risk_score": active_record["risk_score"],
        "victim_id": mask_pii(active_record["victim_account_id"]),
        "mule_account_ids": [mask_pii(v) for v in active_record["mule_account_ids"]],
        "cash_out_account_id": mask_pii(active_record["cash_out_account_id"]),
        "hop_count": active_record["hop_count"],
        "total_value": active_record["total_value"],
        "average_hop_minutes": active_record["average_hop_minutes"],
        "reason_codes": active_record["reason_codes"],
        "rationale": active_record["rationale"],
        "evidence_summary": active_record["evidence_summary"],
    }

    evidence_path = export_dir / f"{active_record['alert_id']}_evidence.json"
    evidence_path.write_text(json.dumps(evidence_payload, indent=2), encoding="utf-8")

    summary_df = pd.DataFrame([{**active_record}])
    summary_df["victim_account_id"] = summary_df["victim_account_id"].apply(mask_pii)
    summary_df["cash_out_account_id"] = summary_df["cash_out_account_id"].apply(mask_pii)
    summary_df["mule_account_ids"] = summary_df["mule_account_ids"].apply(
        lambda ids: " -> ".join(mask_pii(v) for v in ids)
    )
    summary_path = export_dir / "mule_chain_alerts_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"✅ Case evidence exported: {evidence_path}")
    print(f"📊 Masked summary exported: {summary_path}")